# Lab 1 — Can We Trust the Data?
## Data Cleaning and Pipeline Integrity

**Scenario:** A station wants a Precision Recruiting Assistant, but its CRM export contains duplicate engagements, school-name variants, missing identifiers, inconsistent dates, and impossible funnel values.

Your job is to turn unreliable activity records into an auditable school summary.

**Learning goals**

- Profile a dataset before modeling.
- Resolve entities without silently merging the wrong records.
- validate funnel logic: `contacts ≥ appointments ≥ qualified ≥ contracts`.
- Produce a clean table with explicit data-quality flags.

*This is a first-pass scaffold. The final version will be expanded after Labs 2 and 3 stabilize.*

> **Completed instructor version.** Exercise values and functions are filled in, self-checks are executed, and explanations follow each solution. 

> **Use your coding assistant as a teammate.** Give it the current cell, the self-check output, and the goal. Ask it to explain the smallest useful change rather than rewriting the notebook.

Suggested prompt:

> I am working in a classroom Jupyter notebook. Explain what this self-check is testing, then suggest the smallest edit to the marked variables. Do not change the data or the test.

In [1]:
from IPython.display import display, Markdown

def check(name, condition, hint=""):
    try:
        passed = bool(condition)
    except Exception as exc:
        passed = False
        hint = f"{hint} ({type(exc).__name__}: {exc})"
    icon = "✅" if passed else "❌"
    print(f"{icon} {name}")
    if not passed and hint:
        print(f"   Hint: {hint}")
    return passed

def mission_header(text):
    display(Markdown(f"> **Mission checkpoint:** {text}"))

In [2]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)

raw = pd.DataFrame([
    ["E001", "Jefferson HS",       "2026-01-12", 40, 14, 9, 5, 8],
    ["E002", "JEFFERSON HIGH",     "01/20/2026", 32, 11, 7, 4, 6],
    ["E002", "JEFFERSON HIGH",     "01/20/2026", 32, 11, 7, 4, 6],  # duplicate
    ["E003", "Jefferson High School", "2026/02/03", 25, 10, 12, 3, 7], # impossible
    ["E004", "Lincoln High",       "2026-01-15", 55, 19, 8, 3, 9],
    ["E005", "LINCOLN HS",         "not recorded", 31, 10, 6, 2, 5],
    ["E006", "Washington High",    "2025-12-11", 35, 12, 9, 6, 8],
    ["E007", "Washington H.S.",    "2026-02-18", 30, 9, 7, 5, 7],
    ["E008", "Roosevelt High",     "2026-01-22", 28, 8, 5, 3, 6],
    [None,   "Roosevelt High",      "2026-02-14", 20, 7, 4, 2, 5],
    ["E010", "North County Tech",  "2026-02-28", -4, 5, 3, 1, 4],     # impossible
    ["E011", None,                  "2026-03-01", 18, 6, 4, 2, 4],
], columns=[
    "engagement_id", "school_name", "event_date", "contacts",
    "appointments", "qualified", "contracts", "recruiter_hours"
])

raw

,engagement_id,school_name,event_date,contacts,appointments,qualified,contracts,recruiter_hours
0,E001,Jefferson HS,2026-01-12,40,14,9,5,8
1,E002,JEFFERSON HIGH,01/20/2026,32,11,7,4,6
2,E002,JEFFERSON HIGH,01/20/2026,32,11,7,4,6
3,E003,Jefferson High School,2026/02/03,25,10,12,3,7
4,E004,Lincoln High,2026-01-15,55,19,8,3,9
5,E005,LINCOLN HS,not recorded,31,10,6,2,5
6,E006,Washington High,2025-12-11,35,12,9,6,8
7,E007,Washington H.S.,2026-02-18,30,9,7,5,7
8,E008,Roosevelt High,2026-01-22,28,8,5,3,6
9,None,Roosevelt High,2026-02-14,20,7,4,2,5


## 1. Profile before fixing

Pause and predict: how many rows are duplicated? Which columns have missing values? Which rows violate the funnel?

In [3]:
profile = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "unique": raw.nunique(dropna=True),
})
display(profile)
print("Exact duplicate rows:", raw.duplicated().sum())

,dtype,missing,unique
engagement_id,object,1,10
school_name,object,1,9
event_date,object,0,11
contacts,int64,0,11
appointments,int64,0,10
qualified,int64,0,8
contracts,int64,0,6
recruiter_hours,int64,0,6


Exact duplicate rows: 1


## 2. Resolve school identities

Edit only `NAME_MAP`. Use one canonical name for each school. Do not use fuzzy matching blindly: similar names are not always the same entity.

In [4]:
NAME_MAP = {
    "Jefferson HS": "Jefferson High",
    "JEFFERSON HIGH": "Jefferson High",
    "Jefferson High School": "Jefferson High",
    "LINCOLN HS": "Lincoln High",
    "Washington H.S.": "Washington High",
}

clean = raw.copy()
clean["school_name_clean"] = clean["school_name"].replace(NAME_MAP)
clean[["school_name", "school_name_clean"]].drop_duplicates()

,school_name,school_name_clean
0,Jefferson HS,Jefferson High
1,JEFFERSON HIGH,Jefferson High
3,Jefferson High School,Jefferson High
4,Lincoln High,Lincoln High
5,LINCOLN HS,Lincoln High
6,Washington High,Washington High
7,Washington H.S.,Washington High
8,Roosevelt High,Roosevelt High
10,North County Tech,North County Tech
11,None,None


### Solution explanation — entity resolution

Known variants map to a canonical school name before aggregation. This is an explicit, auditable mapping rather than an automatic fuzzy merge, because similarly named schools may be distinct entities.

In [5]:
expected_schools = {
    "Jefferson High", "Lincoln High", "Washington High",
    "Roosevelt High", "North County Tech"
}
observed_schools = set(clean["school_name_clean"].dropna())
check(
    "Known school variants resolve to five canonical schools",
    observed_schools == expected_schools,
    "Map all Jefferson, Lincoln, and Washington variants. Leave missing names missing."
)

✅ Known school variants resolve to five canonical schools


True

## 3. Parse, deduplicate, and validate

Fill the marked choices. Keep rejected records in an audit table; never make them disappear without explanation.

In [6]:
REMOVE_DUPLICATE_IDS = True  # Remove the verified duplicate E002 record
INVALID_DATE_POLICY = "flag"  # Preserve the row and its audit flag

clean["event_date_clean"] = pd.to_datetime(
    clean["event_date"], errors="coerce", format="mixed"
)

if REMOVE_DUPLICATE_IDS:
    clean = clean.drop_duplicates(subset="engagement_id", keep="first")

clean["missing_key"] = clean["engagement_id"].isna() | clean["school_name_clean"].isna()
clean["invalid_date"] = clean["event_date_clean"].isna()
clean["negative_value"] = (clean[["contacts", "appointments", "qualified", "contracts", "recruiter_hours"]] < 0).any(axis=1)
clean["invalid_funnel"] = ~(
    (clean["contacts"] >= clean["appointments"])
    & (clean["appointments"] >= clean["qualified"])
    & (clean["qualified"] >= clean["contracts"])
)
clean["is_valid"] = ~clean[["missing_key", "invalid_date", "negative_value", "invalid_funnel"]].any(axis=1)

clean[["engagement_id", "school_name_clean", "is_valid", "missing_key", "invalid_date", "negative_value", "invalid_funnel"]]

,engagement_id,school_name_clean,is_valid,missing_key,invalid_date,negative_value,invalid_funnel
0,E001,Jefferson High,True,False,False,False,False
1,E002,Jefferson High,True,False,False,False,False
3,E003,Jefferson High,False,False,False,False,True
4,E004,Lincoln High,True,False,False,False,False
5,E005,Lincoln High,False,False,True,False,False
6,E006,Washington High,True,False,False,False,False
7,E007,Washington High,True,False,False,False,False
8,E008,Roosevelt High,True,False,False,False,False
9,None,Roosevelt High,False,True,False,False,False
10,E010,North County Tech,False,False,False,True,True


### Solution explanation — deduplication and validation

The repeated `E002` identifier is a verified duplicate, so one copy is retained. Invalid dates, missing keys, negative values, and impossible funnel order are stored as separate flags. The original problem remains visible for audit and remediation.

In [7]:
check("Duplicate engagement IDs are removed", clean["engagement_id"].dropna().is_unique,
      "Set REMOVE_DUPLICATE_IDS after verifying which record to keep.")
check("Dates are parsed and invalid dates are flagged", clean["invalid_date"].sum() == 1,
      "Use errors='coerce' and preserve an invalid-date flag.")
check("At least two impossible records are detected", (~clean["is_valid"]).sum() >= 2,
      "Check missing keys, negative values, dates, and funnel order.")

✅ Duplicate engagement IDs are removed
✅ Dates are parsed and invalid dates are flagged
✅ At least two impossible records are detected


True

## 4. Produce the model-ready school summary

Aggregate only valid rows. Add a transparent quality measure based on all source records, not just the records that survived.

In [8]:
# Aggregate valid records while retaining quality evidence from the raw source.
valid = clean.loc[clean["is_valid"]].copy()

school_summary = (
    valid.groupby("school_name_clean", as_index=False)
    .agg(
        recruiter_hours=("recruiter_hours", "sum"),
        contacts=("contacts", "sum"),
        appointments=("appointments", "sum"),
        qualified=("qualified", "sum"),
        contracts=("contracts", "sum"),
    )
    .rename(columns={"school_name_clean": "school_name"})
)

quality = clean.groupby("school_name_clean")["is_valid"].mean().rename("data_quality")
school_summary = school_summary.merge(quality, left_on="school_name", right_index=True, how="left")
school_summary

,school_name,recruiter_hours,contacts,appointments,qualified,contracts,data_quality
0,Jefferson High,14,72,25,16,9,0.666667
1,Lincoln High,9,55,19,8,3,0.500000
2,Roosevelt High,6,28,8,5,3,0.500000
3,Washington High,15,65,21,16,11,1.000000


### Solution explanation — model-ready aggregation

Only valid engagement rows contribute to outcome totals. `data_quality` is calculated from all source rows for the school, including rejected records, so cleaning does not hide upstream reliability problems.

In [9]:
check("Summary has one row per school", school_summary["school_name"].is_unique)
check("Summary includes downstream outcomes", {"qualified", "contracts"}.issubset(school_summary.columns))
check("Quality scores stay between 0 and 1", school_summary["data_quality"].between(0, 1).all())

✅ Summary has one row per school
✅ Summary includes downstream outcomes
✅ Quality scores stay between 0 and 1


True

## Handoff to Lab 2

The recommendation system should consume a table like `school_summary`, plus operational fields such as access and distance.

**Reflection:** Which errors should block a recommendation? Which should merely reduce confidence?